# Local Preprocessing: Urban Street Tree Classification

**Dataset:** Urban Street Tree Classification
**Environment:** Local Laptop (Windows / macOS / Linux)


## Local Directory Structure

This notebook expects the following folder structure in your current working directory:

```text
project_root/
├── Dan_preprocessing_local.ipynb   (this notebook)
├── data/
│   ├── tree/                       (your downloaded dataset)
│   │   ├── train/
│   │   ├── val/
│   │   └── test/
│   └── clean_tree/                 (generated by this notebook)
└── outputs/
    └── near_duplicate_review/      (generated images for review)
```

## Step 1: Install Dependencies

Run the cell below to ensure you have all required packages installed locally.

In [1]:
import sys
import subprocess

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Core libraries
install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("matplotlib")
install_if_missing("Pillow")
install_if_missing("torch")
install_if_missing("torchvision")

# Specific library for near-duplicate detection
install_if_missing("imagehash")

print("All dependencies are ready.")

Installing pandas...



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Installing matplotlib...



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Installing Pillow...


...


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Installing torchvision...



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Installing imagehash...
All dependencies are ready.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 1. Imports, Reproducibility, and Local Paths

In [2]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import hashlib
import os
import random
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

import imagehash

warnings.filterwarnings("ignore")

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Local Directory Configuration ---
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

# You placed the dataset directly in ./data/tree
DATASET_ROOT = DATA_DIR / "tree" 

CLEAN_ROOT = DATA_DIR / "clean_tree"
NEAR_REVIEW_DIR = OUTPUT_DIR / "near_duplicate_review"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
EXPECTED_NUM_CLASSES = 23

# Create necessary directories
for d in [DATA_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Base directory  :", BASE_DIR)
print("Dataset root    :", DATASET_ROOT)
print("Clean output    :", CLEAN_ROOT)

assert DATASET_ROOT.exists(), f"\nERROR: Dataset not found at {DATASET_ROOT}.\nPlease ensure the 'tree' folder is inside './data/'."

SOURCE_TRAIN_DIR = DATASET_ROOT / "train"
SOURCE_VAL_DIR = DATASET_ROOT / "val"
SOURCE_TEST_DIR = DATASET_ROOT / "test"

assert SOURCE_TRAIN_DIR.exists(), f"Missing 'train' folder in {DATASET_ROOT}"
assert SOURCE_VAL_DIR.exists(), f"Missing 'val' folder in {DATASET_ROOT}"
assert SOURCE_TEST_DIR.exists(), f"Missing 'test' folder in {DATASET_ROOT}"

print("\n✓ Local dataset structure verified.")

Base directory  : /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI
Dataset root    : /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/data/tree
Clean output    : /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/data/clean_tree

✓ Local dataset structure verified.


## 2. Dataset Structure Verification

In [3]:
def get_classes(directory):
    return sorted([d.name for d in Path(directory).iterdir() if d.is_dir()])

train_classes = get_classes(SOURCE_TRAIN_DIR)
val_classes = get_classes(SOURCE_VAL_DIR)
test_classes = get_classes(SOURCE_TEST_DIR)

print("Number of classes:")
print("Train:", len(train_classes))
print("Val  :", len(val_classes))
print("Test :", len(test_classes))

assert len(train_classes) == EXPECTED_NUM_CLASSES
assert train_classes == val_classes
assert train_classes == test_classes

print("\nClass sets are identical across train / val / test.")

Number of classes:
Train: 23
Val  : 23
Test : 23

Class sets are identical across train / val / test.


## 3. Collect Metadata

In [6]:
def collect_images(split_name, split_dir):
    records = []
    for class_dir in sorted(Path(split_dir).iterdir()):
        if not class_dir.is_dir():
            continue
        for file_path in sorted(class_dir.iterdir()):
            if not file_path.is_file():
                continue
            if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            records.append({
                "split": split_name,
                "class": class_dir.name,
                "filename": file_path.name,
                "path": str(file_path),
            })
    return records

all_records = []
for split_name, split_dir in {"train": SOURCE_TRAIN_DIR, "val": SOURCE_VAL_DIR, "test": SOURCE_TEST_DIR}.items():
    all_records.extend(collect_images(split_name, split_dir))

all_records_df = pd.DataFrame(all_records)
print("Total images found:", len(all_records_df))
display(all_records_df.groupby("split").size().rename("images").to_frame())

Total images found: 4804


,images
split,
test,472
train,3850
val,482


## 4. Image Integrity Check

In [7]:
integrity_results = []
for record in all_records:
    path = Path(record["path"])
    result = {
        "split": record["split"], "class": record["class"], "filename": record["filename"],
        "path": str(path), "valid": False, "format": None, "mode": None,
        "width": None, "height": None, "aspect_ratio": None, "error": None,
    }
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            width, height = img.size
            result["valid"] = True
            result["format"] = img.format
            result["mode"] = img.mode
            result["width"] = width
            result["height"] = height
            result["aspect_ratio"] = width / height if height > 0 else None
    except Exception as e:
        result["error"] = str(e)
    integrity_results.append(result)

integrity_df = pd.DataFrame(integrity_results)
invalid_df = integrity_df[~integrity_df["valid"]]

print(f"Valid images  : {integrity_df['valid'].sum()}")
print(f"Invalid images: {len(invalid_df)}")

Valid images  : 4804
Invalid images: 0


## 5. Exact Duplicate Detection (SHA-256)

In [8]:
valid_df = integrity_df[integrity_df["valid"]].copy()

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

sha_records = []
for _, row in valid_df.iterrows():
    path = row["path"]
    sha_records.append({
        "path": path, "split": row["split"], "class": row["class"],
        "filename": row["filename"], "sha256": sha256_file(path)
    })

hash_df = pd.DataFrame(sha_records)
print("SHA-256 hashes computed:", len(hash_df))

SHA-256 hashes computed: 4804


## 6. Cross-Split Exact Duplicates

In [9]:
cross_split_exact_groups = []
for sha, group in hash_df.groupby("sha256"):
    if group["split"].nunique() > 1:
        cross_split_exact_groups.append(group)

if cross_split_exact_groups:
    cross_split_exact_df = pd.concat(cross_split_exact_groups, ignore_index=True)
else:
    cross_split_exact_df = pd.DataFrame(columns=hash_df.columns)

print("Files involved in cross-split exact duplicates:", len(cross_split_exact_df))

Files involved in cross-split exact duplicates: 23


## 7. Build Exact-Duplicate Removal Manifest

In [10]:
exact_removals = []
seen_exact_paths = set()

def normalize_path(path):
    return os.path.normpath(str(path))

def add_exact_removal(row, reason):
    path = normalize_path(row["path"])
    if path not in seen_exact_paths:
        seen_exact_paths.add(path)
        exact_removals.append({
            "split": row["split"], "class": row["class"],
            "filename": row["filename"], "path": path, "reason": reason
        })

for _, group in cross_split_exact_df.groupby("sha256"):
    splits = set(group["split"])
    if "train" in splits:
        for _, row in group.iterrows():
            if row["split"] != "train":
                add_exact_removal(row, "exact_duplicate_keep_train_remove_eval")
    elif splits == {"val", "test"}:
        for _, row in group.iterrows():
            if row["split"] == "val":
                add_exact_removal(row, "exact_duplicate_keep_test_remove_val")

exact_removals_df = pd.DataFrame(exact_removals)
print("Exact duplicate removals:", len(exact_removals_df))

Exact duplicate removals: 10


## 8. Perceptual Hashing for Near-Duplicates

In [11]:
phash_records = []
for _, row in valid_df.iterrows():
    try:
        with Image.open(row["path"]) as img:
            phash_value = imagehash.phash(img.convert("RGB"))
            phash_records.append({
                "path": row["path"], "split": row["split"], "class": row["class"],
                "filename": row["filename"], "phash": phash_value
            })
    except Exception as e:
        print(f"Could not hash {row['path']}: {e}")

phash_df = pd.DataFrame(phash_records)
print("Perceptual hashes computed:", len(phash_df))

Perceptual hashes computed: 4804


## 9. Cross-Split Near-Duplicate Screening

In [12]:
near_duplicate_rows = []
phash_records_list = phash_df.to_dict("records")

print("Comparing pairs...")
for i, j in combinations(range(len(phash_records_list)), 2):
    a = phash_records_list[i]
    b = phash_records_list[j]
    if a["split"] == b["split"]:
        continue
    distance = a["phash"] - b["phash"]
    if distance <= 12:  # Candidate threshold
        near_duplicate_rows.append({
            "distance": int(distance),
            "split_1": a["split"], "class_1": a["class"], "path_1": a["path"],
            "split_2": b["split"], "class_2": b["class"], "path_2": b["path"],
        })

near_duplicates_df = pd.DataFrame(near_duplicate_rows)
if len(near_duplicates_df) > 0:
    near_duplicates_df = near_duplicates_df.sort_values("distance")
print("Cross-split near-duplicate candidates (distance <= 12):", len(near_duplicates_df))

Comparing pairs...
Cross-split near-duplicate candidates (distance <= 12): 188


## 10. Export Near-Duplicate Pairs for Manual Review

In [13]:
if NEAR_REVIEW_DIR.exists():
    shutil.rmtree(NEAR_REVIEW_DIR)
NEAR_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

near_review_rows = []
if len(near_duplicates_df) > 0:
    for pair_id, (_, row) in enumerate(near_duplicates_df.iterrows(), start=1):
        class_name = str(row["class_1"]).replace("/", "_")
        distance = int(row["distance"])
        pair_dir = NEAR_REVIEW_DIR / f"pair_{pair_id:04d}_dist_{distance}_{class_name}"
        pair_dir.mkdir(parents=True, exist_ok=True)

        for side in [1, 2]:
            src = Path(row[f"path_{side}"])
            split = row[f"split_{side}"]
            class_side = row[f"class_{side}"]
            dst_name = f"{split}__{class_side}__{src.name}"
            dst = pair_dir / dst_name
            try:
                shutil.copy2(src, dst)
            except Exception as e:
                print(f"Could not copy {src}: {e}")

        split_1, split_2 = row["split_1"], row["split_2"]
        if split_1 == "train" and split_2 in {"val", "test"}:
            suggested_remove = normalize_path(row["path_2"])
            suggested_reason = "near_duplicate_of_train_remove_eval"
        elif split_2 == "train" and split_1 in {"val", "test"}:
            suggested_remove = normalize_path(row["path_1"])
            suggested_reason = "near_duplicate_of_train_remove_eval"
        elif {split_1, split_2} == {"val", "test"}:
            if split_1 == "test":
                suggested_remove = normalize_path(row["path_2"])
            else:
                suggested_remove = normalize_path(row["path_1"])
            suggested_reason = "near_duplicate_keep_test_remove_val"
        else:
            suggested_remove = None
            suggested_reason = "manual_only"

        near_review_rows.append({
            "pair_id": pair_id, "distance": row["distance"],
            "split_1": split_1, "class_1": row["class_1"], "path_1": normalize_path(row["path_1"]),
            "split_2": split_2, "class_2": row["class_2"], "path_2": normalize_path(row["path_2"]),
            "suggested_remove_path": suggested_remove, "suggested_reason": suggested_reason,
        })

near_review_df = pd.DataFrame(near_review_rows)
near_review_df.to_csv(OUTPUT_DIR / "near_duplicate_review.csv", index=False)

print("Review directory:", NEAR_REVIEW_DIR)
print("Review CSV:", OUTPUT_DIR / "near_duplicate_review.csv")

Review directory: /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/outputs/near_duplicate_review
Review CSV: /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/outputs/near_duplicate_review.csv


## 11. Confirm Near-Duplicate Removals

By default, we accept the automated suggestions to remove evaluation leaks.

In [14]:
confirmed_near_remove_set = set()
if len(near_review_df) > 0:
    suggested = near_review_df[near_review_df["suggested_remove_path"].notna()]
    confirmed_near_remove_set = set(suggested["suggested_remove_path"].astype(str))

print("Confirmed near-duplicate removals:", len(confirmed_near_remove_set))

Confirmed near-duplicate removals: 148


## 12. Build the Final Removal Manifest

In [15]:
exact_remove_set = set(exact_removals_df["path"].astype(str)) if not exact_removals_df.empty else set()
all_remove_set = exact_remove_set | confirmed_near_remove_set

print("Exact duplicate exclusions :", len(exact_remove_set))
print("Near-duplicate exclusions  :", len(confirmed_near_remove_set))
print("Total exclusions          :", len(all_remove_set))

removal_manifest_df = pd.DataFrame({"path": list(all_remove_set)})
removal_manifest_df.to_csv(OUTPUT_DIR / "removal_manifest.csv", index=False)

Exact duplicate exclusions : 10
Near-duplicate exclusions  : 148
Total exclusions          : 148


## 13. Build the Clean Dataset

In [17]:
def build_clean_tree(source_root, clean_root, remove_set):
    if clean_root.exists():
        shutil.rmtree(clean_root)
    counts = {}
    for split in ["train", "val", "test"]:
        src_split = source_root / split
        dst_split = clean_root / split
        dst_split.mkdir(parents=True, exist_ok=True)
        split_count = 0
        for class_dir in sorted(src_split.iterdir()):
            if not class_dir.is_dir():
                continue
            dst_class_dir = dst_split / class_dir.name
            dst_class_dir.mkdir(parents=True, exist_ok=True)
            for file_path in sorted(class_dir.iterdir()):
                if not file_path.is_file():
                    continue
                if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                if normalize_path(file_path) in remove_set:
                    continue
                shutil.copy2(file_path, dst_class_dir / file_path.name)
                split_count += 1
        counts[split] = split_count
    return counts

clean_counts = build_clean_tree(
    source_root=DATASET_ROOT,
    clean_root=CLEAN_ROOT,
    remove_set=all_remove_set
)

print("Clean dataset counts:")
for split, count in clean_counts.items():
    print(f"{split.capitalize():5s}: {count:,}")
print("\nClean dataset root:")
print(CLEAN_ROOT)

Clean dataset counts:
Train: 3,850
Val  : 401
Test : 405

Clean dataset root:
/Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/data/clean_tree


## 14. Point the ML Pipeline to the Clean Dataset

In [18]:
TRAIN_DIR = CLEAN_ROOT / "train"
VAL_DIR = CLEAN_ROOT / "val"
TEST_DIR = CLEAN_ROOT / "test"

assert TRAIN_DIR.exists()
assert VAL_DIR.exists()
assert TEST_DIR.exists()

print("Train:", TRAIN_DIR)
print("Val  :", VAL_DIR)
print("Test :", TEST_DIR)

Train: /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/data/clean_tree/train
Val  : /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/data/clean_tree/val
Test : /Users/dandev947366/Desktop/HAMK_CLASSES/AI-App-Class/NeuralNetworks-DILIVI/data/clean_tree/test


## 15. Define Model-Input Preprocessing

In [19]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transforms created.")

Transforms created.


## 16. Build PyTorch ImageFolder Datasets

In [20]:
train_dataset = ImageFolder(root=str(TRAIN_DIR), transform=train_transform)
val_dataset = ImageFolder(root=str(VAL_DIR), transform=eval_transform)
test_dataset = ImageFolder(root=str(TEST_DIR), transform=eval_transform)

print("Dataset sizes:")
print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx
print("\nClass mappings verified identical across splits.")

Dataset sizes:
Train: 3850
Val  : 401
Test : 405

Class mappings verified identical across splits.


## 17. Build Baseline DataLoaders

In [21]:
BATCH_SIZE = 32
# Windows often struggles with multiprocessing in Jupyter/DataLoader, so we force 0 if on Windows
NUM_WORKERS = 0 if os.name == 'nt' else 2  
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

print("DataLoaders created.")
print(f"num_workers set to {NUM_WORKERS} for local OS compatibility.")

DataLoaders created.
num_workers set to 2 for local OS compatibility.


## 18. Verify a Transformed Batch

In [22]:
sample_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
images, labels = next(iter(sample_loader))

print("Batch tensor shape:", images.shape)
print("Label tensor shape:", labels.shape)
print("Value range (approx):", images.min().item(), "to", images.max().item())

assert images.shape[1:] == (3, 224, 224)
print("\n✓ CNN input tensor format is correct.")

Batch tensor shape: torch.Size([8, 3, 224, 224])
Label tensor shape: torch.Size([8])
Value range (approx): -2.1179039478302 to 2.640000104904175

✓ CNN input tensor format is correct.


## 19. Final Preprocessing Audit

In [23]:
print("========== FINAL PREPROCESSING AUDIT ==========")
assert TRAIN_DIR.exists()
assert VAL_DIR.exists()
assert TEST_DIR.exists()
assert len(train_dataset.classes) == EXPECTED_NUM_CLASSES
assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx
assert images.shape[1:] == (3, 224, 224)

print("✓ Dataset root verified")
print("✓ Train / val / test verified")
print("✓ 23 classes verified")
print("✓ Clean dataset verified")
print("✓ Class mapping verified")
print("✓ RGB / 224×224 tensor pipeline verified")
print("✓ Baseline DataLoaders verified")

print("\nTrain images:", len(train_dataset))
print("Val images  :", len(val_dataset))
print("Test images :", len(test_dataset))

print("\n========== PREPROCESSING READY ==========")

========== FINAL PREPROCESSING AUDIT ==========
✓ Dataset root verified
✓ Train / val / test verified
✓ 23 classes verified
✓ Clean dataset verified
✓ Class mapping verified
✓ RGB / 224×224 tensor pipeline verified
✓ Baseline DataLoaders verified

Train images: 3850
Val images  : 401
Test images : 405

========== PREPROCESSING READY ==========
